In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from banyan_sigma import membership_probability


BANYAN_INPUT_COLS = [
    "ra", "dec",
    "pmra", "pmdec",
    "pmra_error", "pmdec_error",
    "parallax", "parallax_error",
    "rv", "rv_error",
]


def _banyan_global_value(output, key, i):
    try:
        x = output[key].iloc[i]
        if isinstance(x, (pd.Series, dict)):
            return x.get("Global", np.nan)
        return x
    except Exception:
        return np.nan


def _flatten_banyan_output(output):
    """
    Convert BANYAN Sigma output into a flat DataFrame.
    """

    rows = []

    for i in range(len(output)):
        row = {
            "BANYAN_YA_PROB": _banyan_global_value(output, "YA_PROB", i),
            "BANYAN_BEST_HYP": _banyan_global_value(output, "BEST_HYP", i),
            "BANYAN_BEST_YA": _banyan_global_value(output, "BEST_YA", i),
            "BANYAN_LIST_PROB_YAS": _banyan_global_value(output, "LIST_PROB_YAS", i),
        }

        try:
            all_prob = output["ALL"].iloc[i]

            if isinstance(all_prob, pd.Series):
                items = all_prob.items()
            elif isinstance(all_prob, dict):
                items = all_prob.items()
            else:
                items = []

            for hyp, prob in items:
                row[f"BANYAN_prob_{hyp}"] = prob

        except Exception:
            pass

        rows.append(row)

    return pd.DataFrame(rows)


def _run_banyan_for_subset(df_sub, use_rv):
    """
    Run BANYAN for a subset of rows.
    df_sub must already have valid required numeric columns.
    """

    kwargs = dict(
        ra=df_sub["ra"].to_numpy(float),
        dec=df_sub["dec"].to_numpy(float),
        pmra=df_sub["pmra"].to_numpy(float),
        pmdec=df_sub["pmdec"].to_numpy(float),
        epmra=df_sub["pmra_error"].to_numpy(float),
        epmdec=df_sub["pmdec_error"].to_numpy(float),
        plx=df_sub["parallax"].to_numpy(float),
        eplx=df_sub["parallax_error"].to_numpy(float),
        use_plx=True,
    )

    if use_rv:
        kwargs["rv"] = df_sub["rv"].to_numpy(float)
        kwargs["erv"] = df_sub["rv_error"].to_numpy(float)
        kwargs["use_rv"] = True

    output = membership_probability(**kwargs)
    df_banyan = _flatten_banyan_output(output)
    df_banyan.index = df_sub.index

    return df_banyan, output


def append_banyan_results_from_existing_cols(
    input_table,
    output_table,
    target_col=None,
    drop_existing_banyan_results=True,
    read_original_as_str=True,
    verbose=True,
):
    """
    Read a CSV table with BANYAN input columns, calculate BANYAN Sigma
    membership probabilities row by row in batch, append results, and save.

    Required columns:
        ra, dec,
        pmra, pmdec,
        pmra_error, pmdec_error,
        parallax, parallax_error

    Optional columns:
        rv, rv_error

    Rows with finite rv and rv_error are calculated with RV.
    Rows without finite rv/rv_error are calculated without RV.
    """

    input_table = Path(input_table)
    output_table = Path(output_table)

    if read_original_as_str:
        df_in = pd.read_csv(input_table, dtype=str)
    else:
        df_in = pd.read_csv(input_table)

    if len(df_in.columns) == 0:
        raise ValueError("Input table has no columns.")

    if target_col is None:
        target_col = df_in.columns[0]

    if target_col not in df_in.columns:
        raise ValueError(f"target_col={target_col} not found in input table.")

    required_cols = [
        "ra", "dec",
        "pmra", "pmdec",
        "pmra_error", "pmdec_error",
        "parallax", "parallax_error",
    ]

    missing_cols = [c for c in required_cols if c not in df_in.columns]
    if missing_cols:
        raise ValueError(f"Missing required BANYAN input columns: {missing_cols}")

    df_work = df_in.copy()

    for c in required_cols:
        df_work[c] = pd.to_numeric(df_work[c], errors="coerce")

    # RV columns are optional
    if "rv" not in df_work.columns:
        df_work["rv"] = np.nan
    if "rv_error" not in df_work.columns:
        df_work["rv_error"] = np.nan

    df_work["rv"] = pd.to_numeric(df_work["rv"], errors="coerce")
    df_work["rv_error"] = pd.to_numeric(df_work["rv_error"], errors="coerce")

    basic_valid = np.isfinite(df_work[required_cols]).all(axis=1)

    # RV usable only if both rv and rv_error are finite
    rv_valid = (
        np.isfinite(df_work["rv"])
        & np.isfinite(df_work["rv_error"])
    )

    valid_with_rv = basic_valid & rv_valid
    valid_without_rv = basic_valid & (~rv_valid)
    invalid = ~basic_valid

    if verbose:
        print(f"[input] {input_table}")
        print(f"[output] {output_table}")
        print(f"[target_col] {target_col}")
        print(f"[n rows] {len(df_in)}")
        print(f"[valid with RV] {int(valid_with_rv.sum())}/{len(df_in)}")
        print(f"[valid without RV] {int(valid_without_rv.sum())}/{len(df_in)}")
        print(f"[invalid] {int(invalid.sum())}/{len(df_in)}")

    # initialize result table
    df_banyan_all = pd.DataFrame(index=df_in.index)
    df_banyan_all["BANYAN_input_ok"] = basic_valid
    df_banyan_all["BANYAN_used_rv"] = False
    df_banyan_all["BANYAN_error"] = ""

    raw_outputs = {}

    # ------------------------------------------------------------
    # 1. Run BANYAN with RV
    # ------------------------------------------------------------
    if valid_with_rv.sum() > 0:
        try:
            df_sub = df_work.loc[valid_with_rv].copy()
            df_banyan_rv, output_rv = _run_banyan_for_subset(df_sub, use_rv=True)
            raw_outputs["with_rv"] = output_rv

            for col in df_banyan_rv.columns:
                if col not in df_banyan_all.columns:
                    df_banyan_all[col] = pd.Series(
                        [pd.NA] * len(df_banyan_all),
                        index=df_banyan_all.index,
                        dtype="object",
                    )

            df_banyan_all.loc[df_banyan_rv.index, df_banyan_rv.columns] = df_banyan_rv
            df_banyan_all.loc[df_banyan_rv.index, "BANYAN_used_rv"] = True

        except Exception as e:
            df_banyan_all.loc[valid_with_rv, "BANYAN_input_ok"] = False
            df_banyan_all.loc[valid_with_rv, "BANYAN_error"] = f"with RV: {type(e).__name__}: {e}"

    # ------------------------------------------------------------
    # 2. Run BANYAN without RV
    # ------------------------------------------------------------
    if valid_without_rv.sum() > 0:
        try:
            df_sub = df_work.loc[valid_without_rv].copy()
            df_banyan_no_rv, output_no_rv = _run_banyan_for_subset(df_sub, use_rv=False)
            raw_outputs["without_rv"] = output_no_rv

            for col in df_banyan_no_rv.columns:
                if col not in df_banyan_all.columns:
                    df_banyan_all[col] = pd.Series(
                        [pd.NA] * len(df_banyan_all),
                        index=df_banyan_all.index,
                        dtype="object",
                    )

            df_banyan_all.loc[df_banyan_no_rv.index, df_banyan_no_rv.columns] = df_banyan_no_rv
            df_banyan_all.loc[df_banyan_no_rv.index, "BANYAN_used_rv"] = False

        except Exception as e:
            df_banyan_all.loc[valid_without_rv, "BANYAN_input_ok"] = False
            df_banyan_all.loc[valid_without_rv, "BANYAN_error"] = f"without RV: {type(e).__name__}: {e}"

    # ------------------------------------------------------------
    # 3. Mark invalid rows
    # ------------------------------------------------------------
    df_banyan_all.loc[invalid, "BANYAN_error"] = "Missing required BANYAN input columns"

    # ------------------------------------------------------------
    # 4. Remove old BANYAN result columns if needed
    # ------------------------------------------------------------
    df_out_base = df_in.copy()

    if drop_existing_banyan_results:
        old_result_cols = [
            c for c in df_out_base.columns
            if c.startswith("BANYAN_")
        ]

        if old_result_cols and verbose:
            print("[drop old BANYAN result columns]")
            print(old_result_cols)

        df_out_base = df_out_base.drop(columns=old_result_cols)

    # ------------------------------------------------------------
    # 5. Append BANYAN results and save
    # ------------------------------------------------------------
    df_out = pd.concat(
        [
            df_out_base.reset_index(drop=True),
            df_banyan_all.reset_index(drop=True),
        ],
        axis=1,
    )

    output_table.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(output_table, index=False)

    if verbose:
        n_ok = int(df_out["BANYAN_input_ok"].sum())
        n_rv = int(df_out["BANYAN_used_rv"].sum())
        print("\nDone.")
        print(f"[saved] {output_table}")
        print(f"[BANYAN calculated] {n_ok}/{len(df_out)}")
        print(f"[used RV] {n_rv}/{len(df_out)}")

    return df_out, raw_outputs

In [2]:
df_banyan, raw_outputs  = append_banyan_results_from_existing_cols(
    input_table='vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo.csv',
    output_table='vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo_banyan.csv',
    target_col="target_name",
    verbose=True,
)

[input] vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo.csv
[output] vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo_banyan.csv
[target_col] target_name
[n rows] 57
[valid with RV] 35/57
[valid without RV] 22/57
[invalid] 0/57

Done.
[saved] vizier_J_A+A_623_A35/Hyades_rough_selection_with_query_data_gtomo_banyan.csv
[BANYAN calculated] 57/57
[used RV] 35/57


In [3]:
df_banyan, raw_outputs  = append_banyan_results_from_existing_cols(
    input_table='vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo.csv',
    output_table='vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo_banyan.csv',
    target_col="target_name",
    verbose=True,
)

[input] vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo.csv
[output] vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo_banyan.csv
[target_col] target_name
[n rows] 71
[valid with RV] 57/71
[valid without RV] 14/71
[invalid] 0/71

Done.
[saved] vizier_J_A+A_628_A66/Praesepe_rough_selection_with_query_data_gtomo_banyan.csv
[BANYAN calculated] 71/71
[used RV] 57/71


In [4]:
df_banyan, raw_outputs  = append_banyan_results_from_existing_cols(
    input_table='vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo.csv',
    output_table='vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo_banyan.csv',
    target_col="target_name",
    verbose=True,
)

[input] vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo.csv
[output] vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo_banyan.csv
[target_col] target_name
[n rows] 75
[valid with RV] 53/75
[valid without RV] 20/75
[invalid] 2/75

Done.
[saved] vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo_banyan.csv
[BANYAN calculated] 73/75
[used RV] 53/75


In [5]:
import pandas as pd
import numpy as np

path = "vizier_J_A+A_628_A66/Pleiades_rough_selection_with_query_data_gtomo_banyan.csv"
df = pd.read_csv(path)

required_cols = [
    "ra", "dec",
    "pmra", "pmdec",
    "pmra_error", "pmdec_error",
    "parallax", "parallax_error",
]

for c in required_cols + ["rv", "rv_error"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

bad = df.loc[
    ~df["BANYAN_input_ok"].astype(bool),
    ["target_name", "BANYAN_error"] + required_cols + ["rv", "rv_error"]
]

bad

,target_name,BANYAN_error,ra,dec,pmra,pmdec,pmra_error,pmdec_error,parallax,parallax_error,rv,rv_error
37,Gaia DR2 65230764996027776,Missing required BANYAN input columns,56.566683,24.189922,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,Gaia DR2 66714384141781760,Missing required BANYAN input columns,56.871251,24.104934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# 	Gaia DR2 65230764996027776
# This is a close-in binary that was not resolved by DR2 but by DR3.

#   Gaia DR2 66714384141781760
# This is a Be star, which do not have proper motion measurements from DR3.